# Micro-Neo v5: fine-tuning di un modello piccolo per azioni di scrittura (IT/EN)

Studente **Qwen2.5-1.5B-Instruct** (o 0.5B). Azioni: *riepilogo, spiegazione, parafrasi, migliora, cambia tono (5), accorcia/allunga, citazione, social (LinkedIn, X, Instagram, Facebook), traduci in inglese*.

**Cosa cambia rispetto alla v4**
- **Dataset a copertura bilanciata**: si parte da `neo_dataset.jsonl` (o il vecchio `leo_dataset.jsonl`) (se c'è), lo si ri-filtra con controlli più severi e si genera con il teacher **solo ciò che manca** per arrivare a `TARGET_PER_COMBO` esempi per ogni (azione, parametro). Niente più generazione "alla cieca".
- **Filtri più duri**: hashtag per piattaforma (Instagram 5-8 + emoji, X ≤2 e ≤280 caratteri, LinkedIn ≤3), niente liste di soli hashtag, niente ripetizioni degenerate, niente quasi-copie dell'input, niente output troncati. Il notebook stampa **perché** scarta gli esempi.
- **Nuova azione `translate`** (IT → EN), presente nel menu di NEO ma non addestrata nella v4.
- **Training più veloce**: `group_by_length`, esempi troppo lunghi scartati invece che troncati (un troncamento taglia l'EOS e insegna a non fermarsi), `MAX_SEQ` più basso, checkpointing opzionale.
- **Valutazione onesta**: stessi controlli su modello base (adapter spento) e modello addestrato, per azione. Si vede se il fine-tuning aiuta davvero.
- **Export completo**: GGUF (llama.cpp) **e** MLC (WebLLM / NEO), più `neo_prompts_reference.json` per verificare che i prompt di NEO coincidano con quelli del training.

> ⚠️ I prompt (cella 3) devono restare **identici** a quelli di `index.html` (`buildPrompt`, `SYSTEM`, `MAXTOK`). Se ne cambi uno, cambialo in entrambi e riaddestra.

**Runtime:** GPU T4. Con il dataset già pronto si genera solo il deficit; con dataset vuoto servono ~1-1,5 h di generazione per ~2000 esempi.

### 1. Installazione

In [ ]:
!pip install -q -U transformers peft accelerate datasets bitsandbytes

### 2. Configurazione

In [ ]:
import os
PRESET  = "qwen15"     # "qwen05" | "qwen15" | "gemma1b"   (per NEO/WebLLM usa Qwen2.5: riusa il .wasm del catalogo)
TEACHER = "qwen3b"     # "qwen3b" (veloce) | "qwen7b" (output migliori, ~2x più lento)

TARGET_PER_COMBO = 120   # esempi desiderati per ogni (azione, parametro): 17 combinazioni (~2000 totali)
OVERGEN          = 1.6   # sovra-generazione per compensare i filtri (abbassa se i log mostrano pochi scarti)
ACTIONS_PER_TEXT = 4     # esempi generati per ogni testo sorgente
IT_SHARE, WIKI_SHARE = 0.7, 0.55   # quota di testi italiani; quota di testi da Wikipedia (il resto è sintetico)
N_EVAL   = 120           # esempi tenuti fuori per la valutazione
MAX_SEQ  = 1280          # token massimi per esempio; quelli più lunghi vengono SCARTATI (mai troncati)
GRAD_CKPT = True         # False = ~30% più veloce se la VRAM basta (prova su qwen05)
NEO_DATASET = next((f for f in ("neo_dataset.jsonl", "leo_dataset.jsonl") if os.path.exists(f)), "neo_dataset.jsonl")   # opzionale (accetta anche il vecchio nome): se esiste viene riusato e completato
SEED = 3407

TEACHERS = {
    "qwen3b": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   # già quantizzato a 4 bit (serve bitsandbytes)
    "qwen7b": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
}
PRESETS = {
    "qwen05":  dict(name="Qwen/Qwen2.5-0.5B-Instruct", lr=2e-4, r=64, epochs=6, bs=8, accum=1),
    "qwen15":  dict(name="Qwen/Qwen2.5-1.5B-Instruct", lr=1e-4, r=32, epochs=3, bs=4, accum=2),
    "gemma1b": dict(name="unsloth/gemma-3-1b-it",      lr=1e-4, r=64, epochs=4, bs=4, accum=2),
}
P = PRESETS[PRESET]
print("Student:", P["name"], "| Teacher:", TEACHERS[TEACHER])

### 3. Prompt condivisi (identici a `index.html` di NEO)

In [ ]:
import json, os, random, re, gc, math, difflib
from collections import Counter, defaultdict
import torch
random.seed(SEED)

SYSTEM = ("Sei un assistente di scrittura che lavora in locale. Rispondi SOLO con il risultato richiesto, "
          "senza premesse, commenti o virgolette di contorno. Usa la stessa lingua del testo dell'utente, "
          "salvo diversa richiesta. Non inventare fatti che non sono nel testo.")

TONES = {
    "professional": "professionale e curato",
    "casual": "informale e colloquiale",
    "enthusiastic": "entusiasta e positivo",
    "informative": "informativo e neutro",
    "creative": "creativo e vivace",
}
PLATFORMS = {
    "linkedin": "un post LinkedIn (tono professionale, 3-5 frasi, massimo 3 hashtag)",
    "x": "un post per X/Twitter (massimo 280 caratteri, massimo 2 hashtag)",
    "instagram": "una didascalia Instagram (tono coinvolgente, qualche emoji, 5-8 hashtag in fondo)",
    "facebook": "un post Facebook (tono amichevole, 2-4 frasi)",
}
# max_new_tokens per azione: usati sia per generare i dati sia da NEO in inferenza
MAXTOK = {"summary": 220, "explain": 320, "paraphrase": 350, "improve": 380, "tone": 380,
          "length": 420, "quote": 90, "social": 200, "translate": 400}

COMBOS = ([("summary", ""), ("explain", ""), ("paraphrase", ""), ("improve", "")]
          + [("tone", t) for t in TONES]
          + [("length", "shorter"), ("length", "longer"), ("quote", "")]
          + [("social", p) for p in PLATFORMS]
          + [("translate", "")])

def build_prompt(action: str, param: str, text: str) -> str:
    if action == "summary":
        ins = "Riassumi il testo seguente in modo chiaro e conciso (un paragrafo breve oppure al massimo 5 punti elenco)."
    elif action == "explain":
        ins = "Spiega il testo seguente in parole semplici, come a chi non conosce l'argomento. Chiarisci i termini difficili."
    elif action == "paraphrase":
        ins = "Parafrasa il testo seguente con parole diverse, mantenendo lo stesso significato e la stessa lunghezza circa."
    elif action == "improve":
        ins = "Migliora il testo seguente: correggi errori e refusi e rendi lo stile più chiaro e scorrevole, senza cambiare il significato."
    elif action == "tone" and param in TONES:
        ins = f"Riscrivi il testo seguente con un tono {TONES[param]}, mantenendo il significato."
    elif action == "length" and param == "shorter":
        ins = "Riscrivi il testo seguente in forma molto più breve, tenendo solo le informazioni essenziali."
    elif action == "length" and param == "longer":
        ins = "Riscrivi il testo seguente in forma più lunga e sviluppata, aggiungendo chiarezza e fluidità ma nessun fatto nuovo."
    elif action == "quote":
        ins = "Dal testo seguente ricava una citazione breve e memorabile (una o due frasi) che ne catturi l'idea centrale."
    elif action == "social" and param in PLATFORMS:
        ins = f"Crea {PLATFORMS[param]} basato sul testo seguente."
    elif action == "translate":
        ins = "Traduci il testo seguente in inglese, mantenendo lo stile originale."
    else:
        raise ValueError("azione non valida")
    return f"{ins}\n\nTESTO:\n\"\"\"\n{text}\n\"\"\""

# Riferimento per verificare che NEO costruisca gli stessi prompt: confronta con i prompt generati da index.html
REF_TEXT = "Testo di prova. Seconda frase."
json.dump({"system": SYSTEM, "maxtok": MAXTOK,
           "prompts": {f"{a}/{p}": build_prompt(a, p, REF_TEXT) for a, p in COMBOS}},
          open("neo_prompts_reference.json", "w", encoding="utf8"), ensure_ascii=False, indent=1)
print(len(COMBOS), "combinazioni | riferimento prompt salvato in neo_prompts_reference.json")

### 4. Funzioni comuni: generazione a batch, rumore, filtri e controlli

In [ ]:
def chat_generate(model, tok, messages_list, max_new_tokens=320, temperature=0.4, batch_size=24,
                  sample=True, tk=None, return_trunc=False):
    """Una risposta per ogni lista di messaggi (ordinati per lunghezza). return_trunc -> [(testo, troncato)]."""
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, **(tk or {})) for m in messages_list]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    results = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        kw = (dict(do_sample=True, temperature=temperature, top_p=0.9, repetition_penalty=1.05) if sample
              else dict(do_sample=False, repetition_penalty=1.05))
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, use_cache=True,
                                 pad_token_id=tok.pad_token_id, **kw)
        gen = out[:, enc["input_ids"].shape[1]:]
        for j, i in enumerate(idx):
            n_tok = int((gen[j] != tok.pad_token_id).sum())
            results[i] = (tok.decode(gen[j], skip_special_tokens=True).strip(), n_tok >= max_new_tokens)
        print(f"\r  {min(s + batch_size, len(order))}/{len(order)}", end="")
    print()
    return results if return_trunc else [r[0] for r in results]

def add_noise(text, p=0.06):
    """Refusi realistici per l'azione 'Migliora'."""
    out = []
    for w in text.split(" "):
        if len(w) > 3 and random.random() < p:
            k = random.randrange(4)
            i = random.randrange(1, len(w) - 1)
            if k == 0:   w = w[:i] + w[i + 1] + w[i] + w[i + 2:]          # scambio
            elif k == 1: w = w[:i] + w[i + 1:]                             # lettera mancante
            elif k == 2: w = w[:i] + w[i] + w[i:]                          # lettera doppia
            else:        w = w.translate(str.maketrans("àèéìòù", "aeeiou"))  # accenti persi
        out.append(w)
    t = " ".join(out)
    return t[0].lower() + t[1:] if random.random() < 0.3 and t else t

PREAMBLE = re.compile(r"^\s*(certo|ecco|sure|here is|here's|naturalmente|va bene|ok[,!. ]|riassunto:|riepilogo:)", re.I)
HASH = re.compile(r"#\w+")
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")

def clean_output(text):
    t = text.strip().strip('"').strip("“”").strip()
    if not t or len(t) > 2500 or PREAMBLE.match(t) or "TESTO:" in t or '"""' in t:
        return None
    return t

IT_W = set("il lo la le gli di del della che per con non una un è sono come più anche nel nella dei delle ma se ci si e ed da al alla ai ancora meno molto tra fra su sul sulla ho ha hai mi ti".split())
EN_W = set("the of and to is that for with as on are this it be from by at or an was were which not".split())   # "in" tolto: ambiguo con l'italiano
def guess_lang(t):
    ws = re.findall(r"[a-zàèéìòù']+", t.lower())
    return "it" if sum(w in IT_W for w in ws) >= sum(w in EN_W for w in ws) else "en"

def repetitive(t):
    """Testo degenere (es. lista infinita di hashtag): troppe parole ripetute."""
    ws = re.findall(r"\w+", t.lower())
    return len(ws) >= 30 and len(set(ws)) / len(ws) < 0.35

def similarity(a, b): return difflib.SequenceMatcher(None, a, b).ratio()

def check(action, param, inp, out):
    """Controlli automatici a regole; ritorna {nome: bool}. Usato per filtrare i dati E per valutare il modello."""
    r = {"non_vuoto": bool(out.strip()), "senza_preambolo": not PREAMBLE.match(out),
         "senza_leak": "TESTO:" not in out, "senza_ripetizioni": not repetitive(out)}
    if action == "translate": r["in_inglese"] = guess_lang(out) == "en"
    else:                     r["stessa_lingua"] = len(out) < 60 or guess_lang(out) == guess_lang(inp)
    n_in, n_out = len(inp), len(out)
    if action == "summary":                        r["più_corto"] = n_out < n_in
    if action == "length" and param == "shorter":  r["più_corto"] = n_out < 0.8 * n_in
    if action == "length" and param == "longer":   r["più_lungo"] = n_out > 1.1 * n_in
    if action == "quote":                          r["breve"] = n_out <= 300
    if action in ("explain", "paraphrase", "tone", "social"):
        r["non_copia"] = similarity(inp, out) < 0.85
    if action == "social":
        n_h = len(HASH.findall(out))
        r["testo_oltre_hashtag"] = len(HASH.sub("", out).split()) >= 8
        if param == "x":         r["max_280"] = n_out <= 280; r["max_2_hashtag"] = n_h <= 2
        if param == "linkedin":  r["max_3_hashtag"] = n_h <= 3
        if param == "instagram": r["hashtag_5_8"] = 5 <= n_h <= 8; r["ha_emoji"] = bool(EMOJI.search(out))
    return r

### 5. Dataset esistente + deficit per combinazione

Se `neo_dataset.jsonl` (o il vecchio `leo_dataset.jsonl`) c'è nella cartella di Colab (pannello *File*), viene caricato e **ri-filtrato con i controlli nuovi**. Poi si calcola quanti esempi mancano per ogni (azione, parametro): solo quelli verranno generati.

In [ ]:
def lang_of(e): return guess_lang(e["input"])

examples = []
if os.path.exists(NEO_DATASET):
    raw = [json.loads(l) for l in open(NEO_DATASET, encoding="utf8")]
    bad = Counter()
    for e in raw:
        f = [k for k, v in check(e["action"], e["param"], e["input"], e["output"]).items() if not v]
        if f: bad.update(f)
        else: examples.append(e)
    print(f"Dataset esistente: {len(raw)} righe, {len(examples)} valide con i controlli nuovi. Motivi scarto: {dict(bad)}")
else:
    print("Nessun dataset esistente: si genera tutto.")

have = Counter((e["action"], e["param"]) for e in examples)
deficit = {c: max(0, TARGET_PER_COMBO - have[c]) for c in COMBOS}
for c in COMBOS: print(f"  {c[0]+('/'+c[1] if c[1] else ''):22s} hai {have[c]:4d}  mancano {deficit[c]:4d}")
NEED = sum(deficit.values())
print("Da generare (prima della sovra-generazione):", NEED)

### 6. Teacher: carica il modello grande (solo se serve generare)

In [ ]:
if NEED > 0:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    t_tok = AutoTokenizer.from_pretrained(TEACHERS[TEACHER])
    t_model = AutoModelForCausalLM.from_pretrained(TEACHERS[TEACHER], device_map="cuda", dtype=torch.float16).eval()
    print("Teacher pronto.")

### 7. Testi sorgente: Wikipedia (IT/EN) + testi sintetici, dimensionati sul deficit

In [ ]:
sources = []   # (sid, lang, kind, text)
if NEED > 0:
    from datasets import load_dataset

    n_texts = math.ceil(NEED * OVERGEN / ACTIONS_PER_TEXT * 1.3)
    n_it = round(n_texts * IT_SHARE); n_en = n_texts - n_it
    plan = {("it", "wiki"): round(n_it * WIKI_SHARE), ("it", "synth"): n_it - round(n_it * WIKI_SHARE),
            ("en", "wiki"): round(n_en * WIKI_SHARE), ("en", "synth"): n_en - round(n_en * WIKI_SHARE)}
    print("Testi sorgente da produrre:", plan)

    def wiki_paragraphs(lang, n):
        """Un paragrafo (250-1100 caratteri) da n articoli casuali. Streaming: niente download completo."""
        try:
            ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
            ds = ds.shuffle(seed=SEED, buffer_size=5000)
            out = []
            for ex in ds:
                paras = [p.strip() for p in ex["text"].split("\n\n") if 250 <= len(p.strip()) <= 1100 and not p.startswith("==")]
                if paras: out.append(random.choice(paras))
                if len(out) >= n: break
            return out
        except Exception as e:
            print(f"[wikipedia {lang}] non disponibile ({e}); uso solo testi sintetici")
            return []

    GENRES = {
     "it": ["email di lavoro", "recensione di un prodotto", "articolo di cronaca", "post di blog", "messaggio informale a un amico",
            "verbale di riunione", "descrizione tecnica", "paragrafo di un saggio", "annuncio immobiliare", "nota personale",
            "comunicato stampa", "risposta a un cliente"],
     "en": ["work email", "product review", "news article", "blog post", "casual message to a friend",
            "meeting minutes", "technical description", "essay paragraph", "press release", "customer support reply"],
    }
    TOPICS = ["viaggi", "cucina", "tecnologia", "salute", "sport", "finanza personale", "scuola e università", "ambiente", "musica",
              "cinema", "lavoro da remoto", "auto elettriche", "intelligenza artificiale", "storia", "salute mentale", "giardinaggio",
              "videogiochi", "moda", "sicurezza informatica", "politica locale", "startup", "fotografia", "animali domestici",
              "libri", "energia rinnovabile", "trasporti pubblici", "turismo", "matematica", "medicina", "arte contemporanea"]

    def synthetic_texts(lang, n):
        L = "italiano" if lang == "it" else "inglese"
        reqs = []
        for _ in range(n):
            g, t, w = random.choice(GENRES[lang]), random.choice(TOPICS), random.choice([60, 90, 130, 180, 240])
            reqs.append([{"role": "system", "content": "Scrivi solo il testo richiesto, senza titoli, commenti o virgolette."},
                         {"role": "user", "content": f"Scrivi in {L} un testo di circa {w} parole: {g} sul tema «{t}». Rendilo realistico e specifico."}])
        outs = chat_generate(t_model, t_tok, reqs, max_new_tokens=380, temperature=0.9, batch_size=32, return_trunc=True)
        return [o for o, trunc in outs if not trunc and 200 <= len(o) <= 1500 and guess_lang(o) == lang]

    sid = max([e["sid"] for e in examples], default=-1) + 1
    for (lang, kind), n in plan.items():
        print(kind, lang, n)
        texts = wiki_paragraphs(lang, n) if kind == "wiki" else synthetic_texts(lang, n)
        for t in texts:
            sources.append((sid, lang, kind, t)); sid += 1
    print("Testi sorgente totali:", len(sources))

### 8. Genera solo il deficit (combinazioni con più bisogno per prime), filtra, unisci

In [ ]:
if NEED > 0:
    def eligible(c, n, lang):
        a, p = c
        if a == "summary": return n >= 300
        if a == "quote":   return n >= 200
        if c == ("length", "shorter"): return n > 250
        if c == ("length", "longer"):  return n < 700
        if a == "translate": return lang == "it"       # il prompt chiede l'inglese
        return True

    need = {c: math.ceil(d * OVERGEN) for c, d in deficit.items()}
    jobs = []   # (sid, lang, action, param, input)
    for sid_, lang, kind, text in random.sample(sources, len(sources)):
        if not any(v > 0 for v in need.values()): break
        cand = sorted((c for c in COMBOS if need[c] > 0 and eligible(c, len(text), lang)),
                      key=lambda c: -(need[c] + random.random()))
        for a, p in cand[:ACTIONS_PER_TEXT]:
            need[(a, p)] -= 1
            jobs.append((sid_, lang, a, p, add_noise(text) if (a == "improve" and random.random() < 0.8) else text))
    print("Richieste al teacher:", len(jobs), "| combinazioni non coperte per mancanza di testi:",
          {f"{a}/{p}": v for (a, p), v in need.items() if v > 0} or "nessuna")

    msgs = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(a, p, inp)}] for _, _, a, p, inp in jobs]
    raw_out = {}
    for act in MAXTOK:
        idx = [i for i, j in enumerate(jobs) if j[2] == act]
        if not idx: continue
        print(f"{act}: {len(idx)}")
        res = chat_generate(t_model, t_tok, [msgs[i] for i in idx], max_new_tokens=MAXTOK[act], batch_size=32, return_trunc=True)
        raw_out.update(zip(idx, res))

    fails, new, seen = Counter(), [], {e["output"] for e in examples}
    for i, (sid_, lang, a, p, inp) in enumerate(jobs):
        o, trunc = raw_out[i]
        if trunc:                 fails["troncato"] += 1; continue
        o = clean_output(o)
        if o is None:             fails["clean_output"] += 1; continue
        if o in seen:             fails["duplicato"] += 1; continue
        bad = [k for k, v in check(a, p, inp, o).items() if not v]
        if bad:                   fails.update(bad); continue
        seen.add(o)
        new.append({"sid": sid_, "action": a, "param": p, "input": inp, "output": o})
    print(f"Nuovi esempi validi: {len(new)} / {len(jobs)}  | scarti per motivo: {dict(fails.most_common())}")
    examples += new

    with open("neo_dataset.jsonl", "w", encoding="utf8") as f:
        for e in examples: f.write(json.dumps(e, ensure_ascii=False) + "\n")
    del t_model, t_tok; gc.collect(); torch.cuda.empty_cache()

have = Counter((e["action"], e["param"]) for e in examples)
print(f"\nDataset finale: {len(examples)} esempi")
for c in COMBOS:
    warn = "  <-- POCHI, rilancia dalla cella 5" if have[c] < 0.6 * TARGET_PER_COMBO else ""
    print(f"  {c[0]+('/'+c[1] if c[1] else ''):22s} {have[c]:4d}{warn}")

### 9. Studente: modello piccolo + LoRA (transformers + peft)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# Gemma 3 è instabile in fp16 -> fp32 (è piccolo); gli altri: bf16 se disponibile, altrimenti fp16 (T4)
dtype = torch.float32 if "gemma" in P["name"] else (torch.bfloat16 if bf16_ok else torch.float16)

tokenizer = AutoTokenizer.from_pretrained(P["name"])
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(P["name"], dtype=dtype, device_map="cuda")
print("Architettura:", model.config.architectures, "| parametri: %.0fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

if GRAD_CKPT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=P["r"], lora_alpha=P["r"], lora_dropout=0.0, bias="none",
                                         target_modules="all-linear", task_type="CAUSAL_LM"))
for p in model.parameters():          # adapter in fp32: necessario con fp16 + GradScaler
    if p.requires_grad: p.data = p.data.float()
model.print_trainable_parameters()

# Il template può avere la modalità "thinking": la disattiviamo, coerentemente in training e inferenza
TK = {"enable_thinking": False} if "enable_thinking" in str(tokenizer.chat_template) else {}
print("Argomenti template:", TK)

### 10. Dataset (loss solo sulla risposta) e training

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

def student_msgs(a, p, inp):
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": build_prompt(a, p, inp)}]

def end_suffix(tok):
    """Testo che il template mette dopo la risposta dell'assistant (es. il token di fine turno)."""
    m = student_msgs("summary", "", "x") + [{"role": "assistant", "content": "AAA"}]
    full = tok.apply_chat_template(m, tokenize=False, **TK)
    return full[full.index("AAA") + 3:].rstrip() or tok.eos_token
END = end_suffix(tokenizer)
print("Suffisso di fine risposta:", repr(END))

def encode(e):
    prompt = tokenizer.apply_chat_template(student_msgs(e["action"], e["param"], e["input"]),
                                           tokenize=False, add_generation_prompt=True, **TK)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    c_ids = tokenizer(e["output"] + END, add_special_tokens=False)["input_ids"]
    ids = p_ids + c_ids
    return {"input_ids": ids, "labels": [-100] * len(p_ids) + c_ids, "attention_mask": [1] * len(ids)}   # niente loss sul prompt

enc_all = [(e, encode(e)) for e in examples]
too_long = sum(len(x["input_ids"]) > MAX_SEQ for _, x in enc_all)
enc_all = [(e, x) for e, x in enc_all if len(x["input_ids"]) <= MAX_SEQ]      # scartati, mai troncati (il troncamento taglia l'EOS)
print(f"Scartati perché più lunghi di {MAX_SEQ} token: {too_long}")

# split per testo sorgente: nessuna variante dello stesso testo sia in train che in eval
sids = sorted({e["sid"] for e, _ in enc_all}); random.shuffle(sids)
eval_sids, n_ev = set(), 0
for s in sids:
    if n_ev >= N_EVAL: break
    eval_sids.add(s); n_ev += sum(e["sid"] == s for e, _ in enc_all)
train_ex = [e for e, _ in enc_all if e["sid"] not in eval_sids]
eval_ex  = [e for e, _ in enc_all if e["sid"] in eval_sids]
train_ds = Dataset.from_list([x for e, x in enc_all if e["sid"] not in eval_sids])
val_ds   = Dataset.from_list([x for e, x in random.sample([t for t in enc_all if t[0]["sid"] in eval_sids], min(60, len(eval_ex)))])
print(f"train={len(train_ds)}  eval={len(eval_ex)}")

# controllo visivo: prompt (mascherato) e risposta (addestrata)
ex = train_ds[0]
print("PROMPT  :", tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])[-350:])
print("RISPOSTA:", tokenizer.decode([l for l in ex["labels"] if l != -100])[:350])

steps = (len(train_ds) // (P["bs"] * P["accum"])) * P["epochs"]
model.config.use_cache = False
trainer = Trainer(
    model=model, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(
        output_dir="outputs", per_device_train_batch_size=P["bs"], gradient_accumulation_steps=P["accum"],
        num_train_epochs=P["epochs"], learning_rate=P["lr"], lr_scheduler_type="cosine", warmup_steps=int(steps * 0.05),
        weight_decay=0.01, optim="adamw_torch", group_by_length=True,          # meno padding -> più veloce
        bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16),
        logging_steps=10, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,   # tiene l'epoca migliore
        report_to="none", seed=SEED,
    ),
)
trainer.train()

### 11. Valutazione: modello base (adapter spento) vs addestrato

Stessi controlli automatici usati per filtrare il dataset, sullo stesso set tenuto fuori. **Il fine-tuning è riuscito se "tutti i controlli" sale molto rispetto al base**, soprattutto sulle azioni social. I controlli misurano la *forma*, non la qualità della lingua: leggi anche gli esempi della cella 12.

In [ ]:
if GRAD_CKPT: model.gradient_checkpointing_disable()
model.config.use_cache = True; model.eval()
eval_msgs = [student_msgs(e["action"], e["param"], e["input"]) for e in eval_ex]
# max_new_tokens per azione come in NEO
def gen_eval():
    out = [None] * len(eval_ex)
    for act in MAXTOK:
        idx = [i for i, e in enumerate(eval_ex) if e["action"] == act]
        if not idx: continue
        res = chat_generate(model, tokenizer, [eval_msgs[i] for i in idx], max_new_tokens=MAXTOK[act],
                            batch_size=16, sample=False, tk=TK)
        for i, r in zip(idx, res): out[i] = r
    return out

tuned = gen_eval()
with model.disable_adapter():
    base = gen_eval()

def score(preds):
    by_check, by_combo = defaultdict(lambda: [0, 0]), defaultdict(lambda: [0, 0])
    for e, o in zip(eval_ex, preds):
        r = check(e["action"], e["param"], e["input"], o)
        for k, v in r.items(): by_check[k][0] += bool(v); by_check[k][1] += 1
        c = e["action"] + ("/" + e["param"] if e["param"] else "")
        by_combo[c][0] += all(r.values()); by_combo[c][1] += 1
    return by_check, by_combo

(cb, ob), (ct, ot) = score(base), score(tuned)
print(f"{'controllo':22s} {'base':>10s} {'addestrato':>12s}")
for k in sorted(ct): print(f"{k:22s} {cb[k][0]:>4d}/{cb[k][1]:<4d}   {ct[k][0]:>4d}/{ct[k][1]:<4d}")
print(f"\n{'azione (tutti i controlli)':28s} {'base':>7s} {'addestrato':>11s}")
for c in sorted(ot): print(f"{c:28s} {ob[c][0]/ob[c][1]:>7.0%} {ot[c][0]/ot[c][1]:>11.0%}   (n={ot[c][1]})")
tot_b = sum(v[0] for v in ob.values()) / sum(v[1] for v in ob.values())
tot_t = sum(v[0] for v in ot.values()) / sum(v[1] for v in ot.values())
print(f"\nTUTTI I CONTROLLI: base {tot_b:.0%} -> addestrato {tot_t:.0%}")

print("\nEsempi che falliscono (addestrato):")
shown = 0
for e, o in zip(eval_ex, tuned):
    bad = [k for k, v in check(e["action"], e["param"], e["input"], o).items() if not v]
    if bad and shown < 6:
        shown += 1; print(f"  [{e['action']}/{e['param']}] {bad}: {o[:160]!r}")

### 12. Esempi a confronto (input → teacher → base → addestrato)

In [ ]:
for i in random.sample(range(len(eval_ex)), min(6, len(eval_ex))):
    e = eval_ex[i]
    print("=" * 90)
    print(f"[{e['action']}{'/' + e['param'] if e['param'] else ''}]  INPUT: {e['input'][:300]}")
    print("--- TEACHER :", e["output"][:400])
    print("--- BASE    :", base[i][:400])
    print("--- STUDENT :", tuned[i][:400])

### 13. Export: merge LoRA → modello unito

In [ ]:
model.save_pretrained("micro_neo_lora"); tokenizer.save_pretrained("micro_neo_lora")   # adapter (leggero)
merged = model.merge_and_unload()
merged.save_pretrained("micro_neo_merged", safe_serialization=True)
tokenizer.save_pretrained("micro_neo_merged")
print("Modello unito salvato in micro_neo_merged")

### 14. GGUF (llama.cpp) — per uso locale con `llama-server`

```bash
llama-server -m micro-neo-q8_0.gguf --jinja --port 8081 -c 2048
```
`q8_0` è la scelta più fedele; `q4_k_m` è circa metà dello spazio con una perdita da verificare.

In [ ]:
!pip install -q gguf sentencepiece
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!mkdir -p micro_neo_gguf
!python llama.cpp/convert_hf_to_gguf.py micro_neo_merged --outtype f16 --outfile micro_neo_gguf/micro-neo-f16.gguf
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize micro_neo_gguf/micro-neo-f16.gguf micro_neo_gguf/micro-neo-q8_0.gguf q8_0
!llama.cpp/build/bin/llama-quantize micro_neo_gguf/micro-neo-f16.gguf micro_neo_gguf/micro-neo-q4_k_m.gguf q4_k_m
!ls -lh micro_neo_gguf

### 15. MLC (WebLLM) — per usarlo in **NEO** nel browser

WebLLM non legge il GGUF: serve il formato MLC. Per Qwen2.5 non devi compilare nulla, perché puoi **riusare il `.wasm` del catalogo** (stessa architettura). I comandi seguono la documentazione MLC-LLM ma **non sono stati eseguiti in questo notebook**: se `pip` non trova le wheel per la versione di Python di Colab, esegui la conversione su Linux/WSL con Python 3.11/3.12.

- `--context-window-size 4096 --prefill-chunk-size 1024` devono corrispondere al `.wasm` riusato (`ctx4k_cs1k`).
- Solo Qwen2.5. Per Gemma serve compilare un `.wasm` proprio (`mlc_llm compile --device webgpu`).

In [ ]:
!pip install -q --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cpu mlc-ai-nightly-cpu
!mlc_llm convert_weight micro_neo_merged --quantization q4f16_1 -o micro-neo-q4f16_1-MLC
!mlc_llm gen_config micro_neo_merged --quantization q4f16_1 --conv-template qwen2 --context-window-size 4096 --prefill-chunk-size 1024 -o micro-neo-q4f16_1-MLC
!ls -lh micro-neo-q4f16_1-MLC

**Poi, in NEO** (*Scegli un modello → + Modello personalizzato*):

| Campo | Valore |
|---|---|
| Nome | `micro-neo-q4f16_1-MLC` |
| URL pesi | il tuo repo Hugging Face (o `models/micro-neo/resolve/main/` se la cartella sta in `D:\leo`) |
| Libreria | `@Qwen2.5-1.5B-Instruct-q4f16_1-MLC` (per `qwen05`: `@Qwen2.5-0.5B-Instruct-q4f16_1-MLC`) |
| Contesto | `4096` |
| shader-f16 | spuntato |

**Verifica finale:** prendi 3-4 input della cella 12, lanciali in NEO con la stessa azione e confronta. Se differiscono molto, controlla `neo_prompts_reference.json` contro i prompt di `index.html`, e riesegui la valutazione sul modello quantizzato (q4 perde qualità).

In [ ]:
# Facoltativo: carica su Hugging Face (il token sta nei "Secrets" di Colab come HF_TOKEN, non scriverlo qui)
# from google.colab import userdata
# from huggingface_hub import HfApi
# HF_REPO = "tuo-utente/micro-neo-q4f16_1-MLC"
# api = HfApi(token=userdata.get("HF_TOKEN"))
# api.create_repo(HF_REPO, exist_ok=True)
# api.upload_folder(folder_path="micro-neo-q4f16_1-MLC", repo_id=HF_REPO)

# Facoltativo: salva su Google Drive (dataset, adapter, GGUF, MLC)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r neo_dataset.jsonl neo_prompts_reference.json micro_neo_lora micro_neo_gguf micro-neo-q4f16_1-MLC /content/drive/MyDrive/